# esnfed — live quickstart

This is a **real Python environment running in your browser** (JupyterLite +
Pyodide) — nothing is installed on your machine and no server is involved.

Run each cell with **Shift+Enter**. Edit anything and re-run. The first cell
installs `esnfed` and may take ~20–40 s the first time.

In [ ]:
%pip install -q esnfed

## Train a single Echo State Network on NARMA-10

In [ ]:
import numpy as np
from esnfed import EchoStateNetwork, datasets, topologies, metrics

u, y = datasets.narma10(1500, rng=0)
u_tr, y_tr, u_te, y_te = datasets.split(u, y, 0.7)

W = topologies.random_reservoir(150, density=0.1, rng=0)
esn = EchoStateNetwork(1, 1, W, spectral_radius=0.9, leaking_rate=1.0, washout=100)
esn.fit(u_tr, y_tr)

pred = esn.predict(u_te)
print('NARMA-10 test NRMSE:', round(metrics.nrmse(y_te[100:], pred[100:]), 4))

## Federate it exactly across 5 clients

Each client shares only the ridge sufficient statistics; the server sums them
and solves once — identical to pooling the data, but it stays local.

In [ ]:
from esnfed import federated

parts = datasets.partition_iid(u_tr, y_tr, n_clients=5, rng=0)
esn_kw = dict(spectral_radius=0.9, leaking_rate=1.0, washout=100)
clients, ref = federated.make_shared_clients(W, parts, input_seed=0, esn_kwargs=esn_kw)

W_out = federated.federated_ridge(clients, ref)
Z_test = ref.harvest(u_te)[ref.washout:]
print('Federated NRMSE:', round(metrics.nrmse(y_te[ref.washout:], Z_test @ W_out), 4))

## Your turn

Try a different reservoir topology, spectral radius, or number of clients —
everything in the [docs](https://daibeal.github.io/esnfed/) works here.

In [ ]:
for kind in ['random', 'small_world', 'scale_free', 'ring']:
    Wk = topologies.make_reservoir(kind, 150, rng=0)
    e = EchoStateNetwork(1, 1, Wk, spectral_radius=0.9, washout=100).fit(u_tr, y_tr)
    print(f'{kind:12s} NRMSE = {metrics.nrmse(y_te[100:], e.predict(u_te)[100:]):.4f}')